In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into DataFrames (store full tables for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# helper to convert timestamp column to timezone-aware datetime series
def ts_series(df, ts_col='timestamp'):
    s = pd.to_datetime(df[ts_col], unit='s', utc=True)
    return s.dt.tz_convert(tz)

# Summaries for each file
# metric.csv
metric_ts = ts_series(metric_df)
metric_row_count = metric_df.shape[0]
metric_unique_cmdb = sorted(metric_df['cmdb_id'].dropna().unique().tolist())
metric_unique_kpis = sorted(metric_df['kpi_name'].dropna().unique().tolist())[:50]
metric_min_ts = metric_ts.min() if metric_row_count>0 else None
metric_max_ts = metric_ts.max() if metric_row_count>0 else None
metric_head = metric_df.head(5)

metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'row_count': metric_row_count,
    'unique_cmdb_ids': metric_unique_cmdb,
    'unique_kpi_names_count_shown': len(metric_unique_kpis),
    'unique_kpi_names_sample': metric_unique_kpis,
    'min_timestamp': metric_min_ts,
    'max_timestamp': metric_max_ts
}])

# trace.csv
trace_ts = ts_series(trace_df)
trace_row_count = trace_df.shape[0]
trace_unique_cmdb = sorted(trace_df['cmdb_id'].dropna().unique().tolist())
trace_unique_traces = sorted(trace_df['trace_name'].dropna().unique().tolist())[:50]
trace_min_ts = trace_ts.min() if trace_row_count>0 else None
trace_max_ts = trace_ts.max() if trace_row_count>0 else None
trace_head = trace_df.head(5)

trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'row_count': trace_row_count,
    'unique_cmdb_ids': trace_unique_cmdb,
    'unique_trace_names_count_shown': len(trace_unique_traces),
    'unique_trace_names_sample': trace_unique_traces,
    'min_timestamp': trace_min_ts,
    'max_timestamp': trace_max_ts
}])

# log.csv
log_ts = ts_series(log_df)
log_row_count = log_df.shape[0]
log_unique_cmdb = sorted(log_df['cmdb_id'].dropna().unique().tolist())
log_unique_logs = sorted(log_df['log_name'].dropna().unique().tolist())[:50]
log_min_ts = log_ts.min() if log_row_count>0 else None
log_max_ts = log_ts.max() if log_row_count>0 else None
log_head = log_df.head(5)

log_summary = pd.DataFrame([{
    'file': 'log.csv',
    'row_count': log_row_count,
    'unique_cmdb_ids': log_unique_cmdb,
    'unique_log_names_count_shown': len(log_unique_logs),
    'unique_log_names_sample': log_unique_logs,
    'min_timestamp': log_min_ts,
    'max_timestamp': log_max_ts
}])

# error_logs.csv (no kpi/trace/log name field)
err_ts = ts_series(error_logs_df)
err_row_count = error_logs_df.shape[0]
err_unique_cmdb = sorted(error_logs_df['cmdb_id'].dropna().unique().tolist())
err_min_ts = err_ts.min() if err_row_count>0 else None
err_max_ts = err_ts.max() if err_row_count>0 else None
error_logs_head = error_logs_df.head(5)

error_logs_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'row_count': err_row_count,
    'unique_cmdb_ids': err_unique_cmdb,
    'min_timestamp': err_min_ts,
    'max_timestamp': err_max_ts
}])

# Final compact outputs (store full tables in variables; display only compact summaries and 5-row previews)
metric_summary, metric_head, trace_summary, trace_head, log_summary, log_head, error_logs_summary, error_logs_head

```
Out[1]:
```
```python
# Compose a concise plain-English summary using previously loaded variables
metric_info = metric_summary.loc[0]
trace_info = trace_summary.loc[0]
log_info = log_summary.loc[0]
err_info = error_logs_summary.loc[0]

def sample_list(lst, n=10):
    if not lst:
        return "[]"
    return "[" + ", ".join(str(x) for x in lst[:n]) + (", ..." if len(lst) > n else "") + "]"

summary_text = (
    "Telemetry files summary (compact):\n\n"
    f"1) metric.csv\n"
    f"   - Total rows: {int(metric_info['row_count'])}\n"
    f"   - Unique cmdb_id values: {metric_info['unique_cmdb_ids']}\n"
    f"   - Unique kpi_name sample (showing up to 10 of {metric_info['unique_kpi_names_count_shown']} shown): "
    f"{sample_list(metric_info['unique_kpi_names_sample'], 10)}\n"
    f"   - Timestamp range (UTC+8): {metric_info['min_timestamp']}  to  {metric_info['max_timestamp']}\n"
    f"   - Top 5 rows preview:\n{metric_head.to_string(index=False)}\n\n"
    f"2) trace.csv\n"
    f"   - Total rows: {int(trace_info['row_count'])}\n"
    f"   - Unique cmdb_id values: {trace_info['unique_cmdb_ids']}\n"
    f"   - Unique trace_name sample (showing up to 10 of {trace_info['unique_trace_names_count_shown']} shown): "
    f"{sample_list(trace_info['unique_trace_names_sample'], 10)}\n"
    f"   - Timestamp range (UTC+8): {trace_info['min_timestamp']}  to  {trace_info['max_timestamp']}\n"
    f"   - Top 5 rows preview:\n{trace_head.to_string(index=False)}\n\n"
    f"3) log.csv\n"
    f"   - Total rows: {int(log_info['row_count'])}\n"
    f"   - Unique cmdb_id values: {log_info['unique_cmdb_ids']}\n"
    f"   - Unique log_name sample (showing up to 10 of {log_info['unique_log_names_count_shown']} shown): "
    f"{sample_list(log_info['unique_log_names_sample'], 10)}\n"
    f"   - Timestamp range (UTC+8): {log_info['min_timestamp']}  to  {log_info['max_timestamp']}\n"
    f"   - Top 5 rows preview:\n{log_head.to_string(index=False)}\n\n"
    f"4) error_logs.csv\n"
    f"   - Total rows: {int(err_info['row_count'])}\n"
    f"   - Unique cmdb_id values: {err_info['unique_cmdb_ids']}\n"
    f"   - Timestamp range (UTC+8): {err_info['min_timestamp']}  to  {err_info['max_timestamp']}\n"
    f"   - Top 5 rows preview:\n{error_logs_head.to_string(index=False)}\n\n"
    "Note: Full tables are stored in kernel variables (metric_df, trace_df, log_df, error_logs_df)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  row_count                                    unique_cmdb_ids  unique_kpi_names_count_shown                            unique_kpi_names_sample             min_timestamp             max_timestamp
0  metric.csv      28196  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...                            50  [Container-DOCKER_CONTAINER_2d16f5b2e830--bcou... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00,     timestamp cmdb_id                                     kpi_name         value
0  1615093200    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1615093200    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  1.635440e+01
2  1615093200    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  5.386598e+08
3  1615093200    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.813454e+07
4  1615093200    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  3.808000e-01,         file  row_count                                    unique_cmdb_ids  unique_trace_names_count_shown                          unique_trace_names_sample             min_timestamp             max_timestamp
0  trace.csv       6861  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...                              50  [trace.from_IG01.duration_mean, trace.from_IG0... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00,     timestamp cmdb_id                       trace_name        value
0  1615093200    IG01         trace.self.duration_mean     0.385561
1  1615093200    IG01          trace.self.duration_p95     1.646450
2  1615093200    IG01             trace.self.row_count  1388.000000
3  1615093200    IG01  trace.to_Tomcat01.duration_mean     0.323755
4  1615093200    IG01   trace.to_Tomcat01.duration_p95     1.137250,       file  row_count                                    unique_cmdb_ids  unique_log_names_count_shown                            unique_log_names_sample             min_timestamp             max_timestamp
0  log.csv        500  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...                             3  [log.error_count, log.gc_pause_max, log.row_co... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00,     timestamp   cmdb_id          log_name   value
0  1615093200   Mysql01   log.error_count    0.00
1  1615093200   Mysql01     log.row_count   10.00
2  1615093200  Tomcat01   log.error_count    0.00
3  1615093200  Tomcat01  log.gc_pause_max    0.06
4  1615093200  Tomcat01     log.row_count  374.00,              file  row_count unique_cmdb_ids min_timestamp max_timestamp
0  error_logs.csv          0              []          None          None, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df, trace_df, log_df already in kernel

# Helper to compute group aggregations:
def aggregate_series(df, group_cols, name_col):
    # Ensure 'value' and 'timestamp' exist
    g = df.groupby(group_cols).agg(
        count = ('value', 'count'),
        p95_value = ('value', lambda x: float(x.dropna().quantile(0.95)) if x.notna().any() else float('nan')),
        min_timestamp = ('timestamp', lambda s: pd.to_datetime(int(s.min()), unit='s', utc=True).tz_convert(tz) if s.notna().any() else pd.NaT),
        max_timestamp = ('timestamp', lambda s: pd.to_datetime(int(s.max()), unit='s', utc=True).tz_convert(tz) if s.notna().any() else pd.NaT)
    ).reset_index()
    # Normalize column names: produce 'cmdb_id' and 'name'
    g = g.rename(columns={group_cols[-1]: 'name'})
    return g

# 1) metric aggregation
metric_agg = aggregate_series(metric_df, ['cmdb_id', 'kpi_name'], 'kpi_name')

# 2) trace aggregation
trace_agg = aggregate_series(trace_df, ['cmdb_id', 'trace_name'], 'trace_name')

# 3) log aggregation
log_agg = aggregate_series(log_df, ['cmdb_id', 'log_name'], 'log_name')

# Prepare compact display tables (top 50 rows sorted by p95_value desc)
metric_out = metric_agg.assign(file='metric')[['file','cmdb_id','name','count','p95_value','min_timestamp','max_timestamp']].sort_values('p95_value', ascending=False).head(50)
trace_out  = trace_agg.assign(file='trace')[['file','cmdb_id','name','count','p95_value','min_timestamp','max_timestamp']].sort_values('p95_value', ascending=False).head(50)
log_out    = log_agg.assign(file='log')[['file','cmdb_id','name','count','p95_value','min_timestamp','max_timestamp']].sort_values('p95_value', ascending=False).head(50)

# Display compact summaries (store full aggregated tables in metric_agg, trace_agg, log_agg)
metric_out, trace_out, log_out

```
Out[2]:
```
# Compose a concise plain-English summary of the aggregated results stored in metric_agg, trace_agg, log_agg
def format_top(df, n=5):
    rows = []
    for _, r in df.head(n).iterrows():
        rows.append(f"{r['cmdb_id']} | {r['name']} | p95={r['p95_value']:.6g} | count={int(r['count'])}")
    return "\n".join(rows) if rows else "No rows"

metric_groups = metric_agg.shape[0]
trace_groups = trace_agg.shape[0]
log_groups = log_agg.shape[0]

summary_text = (
    "Aggregated telemetry summary (compact):\n\n"
    f"metric.csv aggregation:\n"
    f" - Total (cmdb_id, kpi_name) groups: {metric_groups}\n"
    f" - Top 5 groups by global P95 (cmdb_id | kpi_name | p95 | count):\n{format_top(metric_out, 5)}\n\n"
    f"trace.csv aggregation:\n"
    f" - Total (cmdb_id, trace_name) groups: {trace_groups}\n"
    f" - Top 5 groups by global P95 (cmdb_id | trace_name | p95 | count):\n{format_top(trace_out, 5)}\n\n"
    f"log.csv aggregation:\n"
    f" - Total (cmdb_id, log_name) groups: {log_groups}\n"
    f" - Top 5 groups by global P95 (cmdb_id | log_name | p95 | count):\n{format_top(log_out, 5)}\n\n"
    "Notes:\n"
    " - metric top items include high network/Redis and Tomcat request KPIs (very large p95 values).\n"
    " - trace top items are mostly large trace.self.row_count entries for docker and Tomcat nodes.\n"
    " - log top items are mainly log.row_count for apache and Tomcat, with gc_pause_max entries showing small p95 values.\n"
    "Full aggregated tables are stored in kernel variables: metric_agg, trace_agg, log_agg."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(        file   cmdb_id                                               name  count     p95_value             min_timestamp             max_timestamp
879   metric   Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15  5.264499e+09 2021-03-07 13:01:00+08:00 2021-03-07 13:29:00+08:00
880   metric   Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     15  4.762661e+09 2021-03-07 13:01:00+08:00 2021-03-07 13:29:00+08:00
914   metric   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     31  4.632909e+09 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
1018  metric   Redis02                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
907   metric   Redis01                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
...      ...       ...                                                ...    ...           ...                       ...                       ...
1498  metric  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  4.382021e+08 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
1172  metric  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  4.177851e+08 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
1389  metric  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  4.090742e+08 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
163   metric      IG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15  3.815157e+08 2021-03-07 13:01:00+08:00 2021-03-07 13:29:00+08:00
75    metric      IG01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     16  3.803117e+08 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00

[50 rows x 7 columns],       file   cmdb_id                           name  count  p95_value             min_timestamp             max_timestamp
221  trace  dockerB1           trace.self.row_count     30    3614.50 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
236  trace  dockerB2           trace.self.row_count     30    3426.70 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
170  trace  Tomcat04           trace.self.row_count     30    2608.85 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
137  trace  Tomcat02           trace.self.row_count     30    2595.45 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
155  trace  Tomcat03           trace.self.row_count     30    2587.35 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
..     ...       ...                            ...    ...        ...                       ...                       ...
29   trace      IG02    trace.to_Tomcat04.row_count     30     195.20 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
164  trace  Tomcat04      trace.from_IG01.row_count     30     193.85 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
14   trace      IG01    trace.to_Tomcat04.row_count     30     193.85 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
35   trace      MG01  trace.from_Tomcat02.row_count     30     193.30 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
140  trace  Tomcat02        trace.to_MG01.row_count     30     193.30 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00

[50 rows x 7 columns],    file   cmdb_id              name  count  p95_value             min_timestamp             max_timestamp
17  log  apache02     log.row_count     31   770.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
15  log  apache01     log.row_count     31   714.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
13  log  Tomcat04     log.row_count     31   382.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
4   log  Tomcat01     log.row_count     31   381.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
10  log  Tomcat03     log.row_count     31   378.5000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
7   log  Tomcat02     log.row_count     31   375.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
1   log   Mysql01     log.row_count     22     5.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
6   log  Tomcat02  log.gc_pause_max     30     0.1055 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
9   log  Tomcat03  log.gc_pause_max     17     0.1020 2021-03-07 13:01:00+08:00 2021-03-07 13:28:00+08:00
12  log  Tomcat04  log.gc_pause_max     18     0.0900 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
3   log  Tomcat01  log.gc_pause_max     19     0.0900 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
0   log   Mysql01   log.error_count     22     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:29:00+08:00
2   log  Tomcat01   log.error_count     31     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
5   log  Tomcat02   log.error_count     31     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
8   log  Tomcat03   log.error_count     31     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
11  log  Tomcat04   log.error_count     31     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
14  log  apache01   log.error_count     31     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00
16  log  apache02   log.error_count     31     0.0000 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables in kernel: metric_df, trace_df, log_df, metric_agg, trace_agg, log_agg

# Incident window (inclusive) as timezone-aware datetimes
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Allowed components
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Helper: convert df timestamps to tz-aware datetimes (cache to avoid repeated conversion)
def df_with_tz(df):
    if '_ts_tz' not in df.columns:
        df['_ts_tz'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

metric_df = df_with_tz(metric_df)
trace_df  = df_with_tz(trace_df)
log_df    = df_with_tz(log_df)

# Generic function to detect anomalies by joining global p95 from agg table and filtering points in window
def detect_anoms(df, agg_df, df_name_col, agg_name_col='name', file_label='metric'):
    # Prepare agg subset for relevant components
    agg_subset = agg_df[agg_df['cmdb_id'].isin(components)].copy()
    # Ensure p95 numeric
    agg_subset['p95_value'] = pd.to_numeric(agg_subset['p95_value'], errors='coerce')
    # Merge p95 into raw df by cmdb_id and name column
    merged = df.merge(agg_subset[['cmdb_id', agg_name_col, 'p95_value']],
                      left_on=['cmdb_id', df_name_col],
                      right_on=['cmdb_id', agg_name_col],
                      how='inner')
    if merged.shape[0] == 0:
        # empty results
        points = merged[[]]  # empty frame
        groups = pd.DataFrame(columns=['file','cmdb_id','name','earliest_anomaly_timestamp',
                                       'anomaly_count_in_window','max_value_in_window','p95_value','severity_metric'])
        return merged, groups  # merged may be empty
    
    # Filter to incident window using tz-aware timestamp column '_ts_tz'
    in_window = merged[(merged['_ts_tz'] >= window_start) & (merged['_ts_tz'] <= window_end)].copy()
    if in_window.shape[0] == 0:
        # no points in window
        groups = pd.DataFrame(columns=['file','cmdb_id','name','earliest_anomaly_timestamp',
                                       'anomaly_count_in_window','max_value_in_window','p95_value','severity_metric'])
        return in_window, groups
    
    # Mark anomaly points where value >= p95_value (consider NaN p95 -> no anomaly)
    in_window['is_anom'] = False
    mask = in_window['p95_value'].notna() & (in_window['value'] >= in_window['p95_value'])
    in_window.loc[mask, 'is_anom'] = True
    # Keep only anomalous points
    anom_points = in_window[in_window['is_anom']].copy()
    if anom_points.shape[0] == 0:
        groups = pd.DataFrame(columns=['file','cmdb_id','name','earliest_anomaly_timestamp',
                                       'anomaly_count_in_window','max_value_in_window','p95_value','severity_metric'])
        return anom_points, groups
    
    # Aggregate per group (cmdb_id + name column)
    group_cols = ['cmdb_id', df_name_col]
    agg = anom_points.groupby(group_cols).agg(
        earliest_anomaly_timestamp = ('timestamp', lambda s: pd.to_datetime(int(s.min()), unit='s', utc=True).tz_convert(tz)),
        anomaly_count_in_window = ('value', 'count'),
        max_value_in_window = ('value', 'max'),
        p95_value = ('p95_value', 'first')  # p95 is same per group
    ).reset_index()
    # compute severity_metric
    def compute_sev(row):
        p95 = row['p95_value']
        mx  = row['max_value_in_window']
        try:
            if pd.isna(p95) or p95 == 0:
                return float(mx) if not pd.isna(mx) else float('nan')
            return float(mx) / float(p95)
        except Exception:
            return float('nan')
    agg['severity_metric'] = agg.apply(compute_sev, axis=1)
    # Normalize column names: 'name'
    agg = agg.rename(columns={df_name_col: 'name'})
    agg.insert(0, 'file', file_label)
    # Reorder columns
    agg = agg[['file','cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity_metric']]
    return anom_points, agg

# Run detection for metric, trace, log
metric_anom_points, metric_anoms = detect_anoms(metric_df, metric_agg, df_name_col='kpi_name', agg_name_col='name', file_label='metric')
trace_anom_points, trace_anoms   = detect_anoms(trace_df, trace_agg, df_name_col='trace_name', agg_name_col='name', file_label='trace')
log_anom_points, log_anoms       = detect_anoms(log_df, log_agg, df_name_col='log_name', agg_name_col='name', file_label='log')

# Ensure output dataframes exist (empty if no anomalies)
for var in ['metric_anoms','trace_anoms','log_anoms','metric_anom_points','trace_anom_points','log_anom_points']:
    if var not in globals():
        globals()[var] = pd.DataFrame()

# Combined top-20 by severity_metric
combined = pd.concat([metric_anoms, trace_anoms, log_anoms], ignore_index=True) if any([not df.empty for df in [metric_anoms, trace_anoms, log_anoms]]) else pd.DataFrame(columns=['file','cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity_metric'])
# Sort and select top 20
combined_top20 = combined.sort_values('severity_metric', ascending=False).head(20)

# Final compact output: combined_top20 (top 20 anomaly groups)
combined_top20

```
Out[3]:
```
# Compose a concise plain-English summary from the anomaly detection results already in kernel
# Reuse existing aggregated/anomaly variables: metric_anoms, trace_anoms, log_anoms, combined_top20

# Ensure variables exist
for v in ['metric_anoms','trace_anoms','log_anoms','combined_top20']:
    if v not in globals():
        globals()[v] = pd.DataFrame()

n_metric = int(metric_anoms.shape[0]) if not metric_anoms.empty else 0
n_trace  = int(trace_anoms.shape[0])  if not trace_anoms.empty  else 0
n_log    = int(log_anoms.shape[0])    if not log_anoms.empty    else 0
total_anoms = n_metric + n_trace + n_log

# Build top entries list (up to 5) for quick highlight
top_rows = []
if not combined_top20.empty:
    for _, r in combined_top20.head(5).iterrows():
        ts = pd.to_datetime(r['earliest_anomaly_timestamp']).tz_convert(pytz.timezone('Asia/Shanghai')) if not pd.isna(r['earliest_anomaly_timestamp']) else None
        ts_str = str(ts) if ts is not None else "N/A"
        top_rows.append(f"{r['file']} | {r['cmdb_id']} | {r['name']} | p95={r['p95_value']:.6g} | max={r['max_value_in_window']:.6g} | severity={r['severity_metric']:.2f} | first={ts_str}")

top_list_text = "\n".join(top_rows) if top_rows else "No anomalies detected in the window."

# Which components dominate the top anomalies
top_cmdbs = combined_top20['cmdb_id'].value_counts().head(6).to_dict() if not combined_top20.empty else {}

summary = (
    f"Anomaly detection summary for window 2021-03-07 13:00:00+08 to 2021-03-07 13:30:00+08:\n\n"
    f"- Total anomaly groups found: {total_anoms} (metric={n_metric}, trace={n_trace}, log={n_log}).\n\n"
    f"- Highest-severity issues (top 5):\n{top_list_text}\n\n"
    f"- Observations:\n"
    f"  * The most severe anomalies are metric groups on MySQL nodes (Mysql01/Mysql02) with severity ~20x above their global P95 for various MySQL internal counters (many single-point spikes).\n"
    f"  * Other notable high-severity metric anomalies include CPU I/O wait on IG02 and Redis02 (severity ~7x), and TCP socket state counts on Tomcat02/MG01 (~6.7x).\n"
    f"  * Disk read/write metrics on apache02 and Mysql01 also appear in the top anomalies (moderate severity ~3–4x).\n\n"
    f"- Top components by anomaly count in the displayed top-20: {top_cmdbs}\n\n"
    f"Notes: Full per-group anomaly tables are cached in kernel variables metric_anoms, trace_anoms, log_anoms; top-20 summary is in combined_top20."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

file   cmdb_id                                               name earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  p95_value  severity_metric
615   metric   Mysql02                 Mysql-MySQL_3306_Handler Read Next  2021-03-07 13:26:00+08:00                        1            10.864400   0.543220        20.000000
485   metric   Mysql01         Mysql-MySQL_3306_max trx lock memory bytes  2021-03-07 13:00:00+08:00                        1          1136.000000  56.800000        20.000000
654   metric   Mysql02      Mysql-MySQL_3306_Innodb os log pending writes  2021-03-07 13:16:00+08:00                        1             1.000000   0.050000        20.000000
617   metric   Mysql02                  Mysql-MySQL_3306_Handler Read Rnd  2021-03-07 13:26:00+08:00                        1             0.627100   0.031355        20.000000
641   metric   Mysql02        Mysql-MySQL_3306_Innodb data pending writes  2021-03-07 13:16:00+08:00                        1             1.000000   0.050000        20.000000
675   metric   Mysql02          Mysql-MySQL_3306_Opened table definitions  2021-03-07 13:26:00+08:00                        1             0.169500   0.008475        20.000000
695   metric   Mysql02                        Mysql-MySQL_3306_Sort Range  2021-03-07 13:26:00+08:00                        1             0.627100   0.031355        20.000000
450   metric   Mysql01                     Mysql-MySQL_3306_Opened Tables  2021-03-07 13:00:00+08:00                        1             0.033300   0.001665        20.000000
701   metric   Mysql02           Mysql-MySQL_3306_Table open cache misses  2021-03-07 13:26:00+08:00                        1             1.169500   0.058475        20.000000
674   metric   Mysql02                     Mysql-MySQL_3306_Opened Tables  2021-03-07 13:26:00+08:00                        1             1.169500   0.058475        20.000000
446   metric   Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified  2021-03-07 13:00:00+08:00                        1             1.000000   0.050000        20.000000
477   metric   Mysql01           Mysql-MySQL_3306_Table open cache misses  2021-03-07 13:00:00+08:00                        1             0.033300   0.001665        20.000000
444   metric   Mysql01               Mysql-MySQL_3306_Max trx rows locked  2021-03-07 13:00:00+08:00                        1             1.000000   0.050000        20.000000
372   metric   Mysql01                        Mysql-MySQL_3306_Com Insert  2021-03-07 13:00:00+08:00                        1             0.266700   0.013335        20.000000
103   metric      IG02                             OSLinux-CPU_CPU_CPUWio  2021-03-07 13:00:00+08:00                        2             1.355200   0.182852         7.411438
933   metric   Redis02                             OSLinux-CPU_CPU_CPUWio  2021-03-07 13:00:00+08:00                        2             0.513400   0.073045         7.028544
1206  metric  Tomcat02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-07 13:01:00+08:00                        1             1.000000   0.150000         6.666667
246   metric      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-07 13:11:00+08:00                        1             1.000000   0.150000         6.666667
1604  metric  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  2021-03-07 13:00:00+08:00                        1            39.200000   9.800000         4.000000
544   metric   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  2021-03-07 13:15:00+08:00                        1             0.133333   0.040000         3.333333```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: metric_anoms, trace_anoms, log_anoms
# Ensure they exist
for v in ['metric_anoms','trace_anoms','log_anoms']:
    if v not in globals():
        globals()[v] = pd.DataFrame()

# Helper to aggregate per-component
def component_aggregate(anoms_df, file_label):
    if anoms_df.empty:
        cols = ['file','cmdb_id','anomalous_group_count','total_anomaly_points',
                'earliest_anomaly_timestamp','max_severity','top_names']
        return pd.DataFrame(columns=cols)
    # Ensure timestamps are timezone-aware datetimes
    if anoms_df['earliest_anomaly_timestamp'].dtype == object:
        anoms_df['earliest_anomaly_timestamp'] = pd.to_datetime(anoms_df['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
    else:
        anoms_df['earliest_anomaly_timestamp'] = pd.to_datetime(anoms_df['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
    # Group by component
    grp = anoms_df.groupby('cmdb_id')
    comp_rows = []
    for cmdb, g in grp:
        anomalous_group_count = int(g['name'].nunique())
        total_anomaly_points = int(g['anomaly_count_in_window'].sum())
        earliest_anomaly_timestamp = g['earliest_anomaly_timestamp'].min()
        max_severity = float(g['severity_metric'].max())
        # top names by severity desc (up to 5)
        top_names = g.sort_values('severity_metric', ascending=False)['name'].head(5).tolist()
        comp_rows.append({
            'file': file_label,
            'cmdb_id': cmdb,
            'anomalous_group_count': anomalous_group_count,
            'total_anomaly_points': total_anomaly_points,
            'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
            'max_severity': max_severity,
            'top_names': top_names
        })
    comp_df = pd.DataFrame(comp_rows)
    return comp_df

# Compute component-level aggregations and store in kernel variables
metric_comp_agg = component_aggregate(metric_anoms, 'metric')
trace_comp_agg  = component_aggregate(trace_anoms,  'trace')
log_comp_agg    = component_aggregate(log_anoms,    'log')

# Combined summary sorted by max_severity desc, limit to top 20
combined_comp = pd.concat([metric_comp_agg, trace_comp_agg, log_comp_agg], ignore_index=True) if any([not df.empty for df in [metric_comp_agg, trace_comp_agg, log_comp_agg]]) else pd.DataFrame(columns=['file','cmdb_id','anomalous_group_count','total_anomaly_points','earliest_anomaly_timestamp','max_severity','top_names'])

combined_comp = combined_comp.sort_values('max_severity', ascending=False).head(20)

# Keep output compact
combined_comp[['file','cmdb_id','anomalous_group_count','total_anomaly_points','earliest_anomaly_timestamp','max_severity','top_names']].head(20)

```
Out[4]:
```
Compact summary (component-level) of anomalies in the incident window 2021-03-07 13:00–13:30 (UTC+8):

Key findings (top components by max_severity)
- Mysql01 (metric)
  - anomalous_group_count: 224
  - total_anomaly_points: 2,354
  - earliest anomaly: 2021-03-07 13:00:00+08:00
  - max_severity: 20.0
  - top metric examples: Mysql-MySQL_3306_Com Insert, Mysql-MySQL_3306_MaxTrxRowsModified, Mysql-MySQL_3306_Table open cache misses

- Mysql02 (metric)
  - anomalous_group_count: 224
  - total_anomaly_points: 2,058
  - earliest anomaly: 2021-03-07 13:00:00+08:00
  - max_severity: 20.0
  - top metric examples: Mysql-MySQL_3306_Handler Read Rnd, Innodb data pending writes, Table open cache misses

- IG02 (metric)
  - anomalous_group_count: 88
  - total_anomaly_points: 650
  - earliest anomaly: 2021-03-07 13:00:00+08:00
  - max_severity: ~7.41
  - top metric examples: OSLinux-CPU_CPU_CPUWio, local disk/network items

- Redis02 (metric)
  - anomalous_group_count: 111
  - total_anomaly_points: 1,375
  - earliest anomaly: 2021-03-07 13:00:00+08:00
  - max_severity: ~7.03
  - top metric examples: CPU WIO, local disk metrics

- MG01 and Tomcat02 (metric)
  - MG01: 92 groups, 618 points, earliest 13:00, max_severity ~6.67 (TCP socket state metrics)
  - Tomcat02: 109 groups, 950 points, earliest 13:01, max_severity ~6.67 (TCP-FIN-WAIT etc.)

Other notable components
- apache02 / apache01: disk-related metric anomalies (apache02 max_severity 4.0; apache01 max_severity ~3.20)
- Tomcat01 / Tomcat03 / Tomcat04: many metric groups with moderate severity (2.0–2.22)
- Trace-level anomalies (lower severity overall): MG02, MG01, Tomcat nodes show trace-group anomalies (max_severity ~2.53 down to ~1.32)
- Log-level anomalies are few and lower severity:
  - Mysql01 (log): 2 anomalous log groups, 26 points, max_severity 2.0 (log.row_count, log.error_count)
  - Tomcat02 (log): 3 groups, 35 points, max_severity ~1.33 (log.gc_pause_max, log.row_count, log.error_count)

Overall interpretation
- The most severe and widespread anomalies are on the MySQL nodes (Mysql01 and Mysql02) with many metric groups spiking up to ~20× their global P95 (single-point spikes dominate).
- Secondary hotspots are IG02 and Redis02 (high CPU I/O wait and disk/network metrics, ~7× P95) and some network/socket state spikes on MG01/Tomcat02 (~6.7×).
- Trace anomalies exist but are lower severity; log anomalies are limited and lower-severity.

Data note
- All times shown are UTC+8. Full component-level aggregated tables are stored in kernel variables metric_comp_agg, trace_comp_agg, log_comp_agg for further inspection.

The original code execution output of IPython Kernel is also provided below for reference:

file   cmdb_id  anomalous_group_count  total_anomaly_points earliest_anomaly_timestamp  max_severity                                          top_names
4   metric   Mysql01                    224                  2354  2021-03-07 13:00:00+08:00     20.000000  [Mysql-MySQL_3306_Com Insert, Mysql-MySQL_3306...
5   metric   Mysql02                    224                  2058  2021-03-07 13:00:00+08:00     20.000000  [Mysql-MySQL_3306_Handler Read Rnd, Mysql-MySQ...
1   metric      IG02                     88                   650  2021-03-07 13:00:00+08:00      7.411438  [OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCAL...
7   metric   Redis02                    111                  1375  2021-03-07 13:00:00+08:00      7.028544  [OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCAL...
2   metric      MG01                     92                   618  2021-03-07 13:00:00+08:00      6.666667  [OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAI...
9   metric  Tomcat02                    109                   950  2021-03-07 13:00:00+08:00      6.666667  [OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT,...
13  metric  apache02                     80                   678  2021-03-07 13:00:00+08:00      4.000000  [OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPe...
6   metric   Redis01                    115                  1370  2021-03-07 13:00:00+08:00      3.333333  [OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBp...
0   metric      IG01                     88                   653  2021-03-07 13:00:00+08:00      3.292872  [OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRT...
12  metric  apache01                     80                   451  2021-03-07 13:00:00+08:00      3.198322  [OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPe...
17   trace      MG02                     39                    82  2021-03-07 13:00:00+08:00      2.534625  [trace.from_dockerB2.duration_mean, trace.from...
8   metric  Tomcat01                    106                   980  2021-03-07 13:00:00+08:00      2.222222  [Tomcat-Threads_7441-"http-nio-8003"_CurrentTh...
10  metric  Tomcat03                    111                   993  2021-03-07 13:00:00+08:00      2.000000  [OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTp...
22     log   Mysql01                      2                    26  2021-03-07 13:00:00+08:00      2.000000                   [log.row_count, log.error_count]
3   metric      MG02                     94                   693  2021-03-07 13:00:00+08:00      1.912695  [OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCAL...
11  metric  Tomcat04                    109                   977  2021-03-07 13:00:00+08:00      1.840071  [OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCAL...
16   trace      MG01                     39                    79  2021-03-07 13:00:00+08:00      1.506087  [trace.from_dockerB2.duration_mean, trace.from...
18   trace  Tomcat01                     18                    35  2021-03-07 13:00:00+08:00      1.425592  [trace.to_MG01.duration_p95, trace.from_IG02.d...
24     log  Tomcat02                      3                    35  2021-03-07 13:00:00+08:00      1.327014  [log.gc_pause_max, log.row_count, log.error_co...
21   trace  Tomcat04                     15                    30  2021-03-07 13:02:00+08:00      1.318904  [trace.self.duration_p95, trace.from_IG01.dura...```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse kernel variables: metric_df, metric_agg
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df has timezone-aware timestamp column _ts_tz
if '_ts_tz' not in metric_df.columns:
    metric_df['_ts_tz'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # Ensure proper dtype and tz
    metric_df['_ts_tz'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target components and incident window
components = ["Mysql01","Mysql02","IG02","Redis02","MG01","Tomcat02"]
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Prepare p95 lookup from metric_agg (agg uses 'cmdb_id' and 'name' where name == kpi_name)
# Ensure p95_value numeric
metric_agg['p95_value'] = pd.to_numeric(metric_agg['p95_value'], errors='coerce')

# Subset metric_agg to only components of interest
p95_lookup = metric_agg[metric_agg['cmdb_id'].isin(components)][['cmdb_id','name','p95_value']].copy()
p95_lookup = p95_lookup.rename(columns={'name':'kpi_name'})

# Merge p95 into metric_df (inner join to keep only KPI groups that have a global p95)
m = metric_df.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='inner')

# Filter to incident window and components (components already applied via p95_lookup merge)
m_window = m[(m['_ts_tz'] >= window_start) & (m['_ts_tz'] <= window_end)].copy()

# Mark anomalous points: value >= p95_value and p95_value not NaN
m_window['p95_value'] = pd.to_numeric(m_window['p95_value'], errors='coerce')
m_window['is_anom'] = m_window['p95_value'].notna() & (m_window['value'] >= m_window['p95_value'])

# Keep only anomalous points
anom_points = m_window[m_window['is_anom']].copy()

# If no anomalous points, create empty metric_faults
if anom_points.empty:
    metric_faults = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length','max_value_in_run','p95_value','severity'])
else:
    # For each (cmdb_id, kpi_name), sort by timestamp and find consecutive runs (diff == 60s)
    anom_points = anom_points.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # compute time diff in seconds within each group
    anom_points['prev_ts'] = anom_points.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
    anom_points['dt'] = anom_points['timestamp'] - anom_points['prev_ts']
    # start of a new run when dt != 60 or prev is NaN
    anom_points['new_run'] = ((anom_points['dt'] != 60) | (anom_points['dt'].isna())).astype(int)
    # assign run id by cumulative sum within group
    anom_points['run_id'] = anom_points.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()

    # Aggregate runs
    run_agg = anom_points.groupby(['cmdb_id','kpi_name','run_id']).agg(
        run_start_timestamp = ('_ts_tz', 'min'),
        run_end_timestamp   = ('_ts_tz', 'max'),
        run_length          = ('timestamp', 'count'),  # number of minutes in run
        max_value_in_run    = ('value', 'max'),
        p95_value           = ('p95_value', 'first')
    ).reset_index(drop=False)

    # Compute severity per rule: severity = max_value_in_run / p95_value if p95_value>0 else max_value_in_run
    def compute_sev(row):
        p95 = row['p95_value']
        mx = row['max_value_in_run']
        try:
            if pd.isna(p95) or p95 == 0:
                return float(mx)
            return float(mx) / float(p95)
        except Exception:
            return float('nan')

    run_agg['severity'] = run_agg.apply(compute_sev, axis=1)

    # Filter runs: keep only runs with run_length >=2 OR severity >= 2.0
    metric_faults = run_agg[(run_agg['run_length'] >= 2) | (run_agg['severity'] >= 2.0)].copy()

    # Normalize timestamp columns to tz-aware strings (keep as tz-aware datetimes)
    # Ensure types are datetime with tz
    metric_faults['run_start_timestamp'] = pd.to_datetime(metric_faults['run_start_timestamp']).dt.tz_convert(tz)
    metric_faults['run_end_timestamp']   = pd.to_datetime(metric_faults['run_end_timestamp']).dt.tz_convert(tz)

    # Select and order columns as requested
    metric_faults = metric_faults[['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length','max_value_in_run','p95_value','severity']]

# Sort metric_faults by severity desc then run_length desc and keep top 50 for display
metric_faults_sorted = metric_faults.sort_values(['severity','run_length'], ascending=[False,False]).reset_index(drop=True)
metric_faults_head = metric_faults_sorted.head(50)

# Store full runs table in kernel variable metric_faults (already set)
metric_faults, metric_faults_head

```
Out[5]:
```
# Compose a plain-English compact summary of consecutive anomaly runs (fault candidates)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_faults exists
if 'metric_faults' not in globals():
    metric_faults = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length','max_value_in_run','p95_value','severity'])

# Focus components
components = ["Mysql01","Mysql02","IG02","Redis02","MG01","Tomcat02"]

mf = metric_faults.copy()
# Ensure timestamps are timezone-aware datetimes
if not mf.empty:
    mf['run_start_timestamp'] = pd.to_datetime(mf['run_start_timestamp']).dt.tz_convert(tz)
    mf['run_end_timestamp']   = pd.to_datetime(mf['run_end_timestamp']).dt.tz_convert(tz)

# Filter to target components
mf_focused = mf[mf['cmdb_id'].isin(components)].copy()

# Aggregate summary stats
total_runs = int(mf_focused.shape[0])
runs_by_component = mf_focused['cmdb_id'].value_counts().to_dict()
total_run_minutes = int(mf_focused['run_length'].sum()) if total_runs>0 else 0
avg_severity = float(mf_focused['severity'].mean()) if total_runs>0 else 0.0

# Top runs by severity then length
top_runs = mf_focused.sort_values(['severity','run_length'], ascending=[False, False]).head(10)

def fmt_row(r):
    start = pd.to_datetime(r['run_start_timestamp']).tz_convert(tz).strftime('%Y-03-07 %H:%M:%S%z') if not pd.isna(r['run_start_timestamp']) else "N/A"
    # Use the stored timestamp string if formatting produced the same date; simpler: use ISO format
    start = pd.to_datetime(r['run_start_timestamp']).tz_convert(tz).isoformat() if not pd.isna(r['run_start_timestamp']) else "N/A"
    return f"{r['cmdb_id']} | {r['kpi_name']} | severity={r['severity']:.2f} | length={int(r['run_length'])}min | start={start}"

top_list = "\n".join(fmt_row(r) for _, r in top_runs.iterrows()) if not top_runs.empty else "No runs found"

# Count how many runs are long (>=2 minutes) vs single-minute but high severity (>=2)
long_runs = int(mf_focused[mf_focused['run_length'] >= 2].shape[0])
high_sev_single = int(mf_focused[(mf_focused['run_length'] == 1) & (mf_focused['severity'] >= 2.0)].shape[0])

summary_text = (
    "Fault-candidate runs summary for selected components in window 2021-03-07 13:00–13:30 (UTC+8):\n\n"
    f"- Components analyzed: {components}\n"
    f"- Total qualifying runs (run_length>=2 OR severity>=2): {total_runs}\n"
    f"- Total run-minutes across these runs: {total_run_minutes}\n"
    f"- Runs by component (count): {runs_by_component}\n"
    f"- Long runs (>=2 minutes): {long_runs}; single-minute high-severity runs (length=1 and severity>=2): {high_sev_single}\n"
    f"- Average severity among these runs: {avg_severity:.2f}\n\n"
    "Top fault-candidate runs (by severity, then length) — up to 10 shown:\n"
    f"{top_list}\n\n"
    "Quick interpretation:\n"
    "- Many highest-severity entries are on Mysql01/Mysql02: numerous single-minute spikes with very high severity (~20x P95) across many MySQL counters.\n"
    "- IG02 shows long continuous runs (many 31-minute runs) for JVM memory/OS metrics (severity ~1), indicating sustained resource levels rather than spikes.\n"
    "- Redis02 has short 2-minute runs with severity near 1 (values close to their P95) for Redis counters.\n"
    "- MG01 and Tomcat02 also contribute runs (some short but with elevated severity for network/socket KPIs).\n\n"
    "Full runs table is cached in kernel variable: metric_faults."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name       run_start_timestamp         run_end_timestamp  run_length  max_value_in_run     p95_value  severity
0         IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00          31      2.040136e+09  2.040136e+09  1.000000
4         IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00          31      5.805958e+07  5.805958e+07  1.000000
7         IG02                    JVM-Runtime_7778_JVM_JVM_Uptime 2021-03-07 13:28:00+08:00 2021-03-07 13:29:00+08:00           2      1.741830e+05  1.741230e+05  1.000345
14        IG02                         OSLinux-CPU_CPU_CPUCpuUtil 2021-03-07 13:00:00+08:00 2021-03-07 13:01:00+08:00           2      2.749000e+01  2.630934e+01  1.044876
22        IG02                             OSLinux-CPU_CPU_CPUWio 2021-03-07 13:00:00+08:00 2021-03-07 13:00:00+08:00           1      1.355200e+00  1.828525e-01  7.411438
...        ...                                                ...                       ...                       ...         ...               ...           ...       ...
4670  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00          31      0.000000e+00  0.000000e+00  0.000000
4671  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00          31      0.000000e+00  0.000000e+00  0.000000
4672  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00          31      0.000000e+00  0.000000e+00  0.000000
4673  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr... 2021-03-07 13:00:00+08:00 2021-03-07 13:03:00+08:00           4      2.820000e+02  2.810000e+02  1.003559
4677  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads... 2021-03-07 13:00:00+08:00 2021-03-07 13:30:00+08:00          31      5.000000e+02  5.000000e+02  1.000000

[1640 rows x 8 columns],     cmdb_id                                           kpi_name       run_start_timestamp         run_end_timestamp  run_length  max_value_in_run     p95_value   severity
0   Mysql01                        Mysql-MySQL_3306_Com Insert 2021-03-07 13:00:00+08:00 2021-03-07 13:00:00+08:00           1      2.667000e-01  1.333500e-02  20.000000
1   Mysql01               Mysql-MySQL_3306_Max trx rows locked 2021-03-07 13:00:00+08:00 2021-03-07 13:00:00+08:00           1      1.000000e+00  5.000000e-02  20.000000
2   Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified 2021-03-07 13:00:00+08:00 2021-03-07 13:00:00+08:00           1      1.000000e+00  5.000000e-02  20.000000
3   Mysql01                     Mysql-MySQL_3306_Opened Tables 2021-03-07 13:00:00+08:00 2021-03-07 13:00:00+08:00           1      3.330000e-02  1.665000e-03  20.000000
4   Mysql01           Mysql-MySQL_3306_Table open cache misses 2021-03-07 13:00:00+08:00 2021-03-07 13:00:00+08:00           1      3.330000e-02  1.665000e-03  20.000000
..      ...                                                ...                       ...                       ...         ...               ...           ...        ...
45  Redis02  redis-Redis_6379_Redis  (rdb_changes_since_las... 2021-03-07 13:29:00+08:00 2021-03-07 13:30:00+08:00           2      8.389195e+08  8.388783e+08   1.000049
46  Redis02            redis-Redis_6379_Redis  (keyspace_hits) 2021-03-07 13:29:00+08:00 2021-03-07 13:30:00+08:00           2      2.698166e+09  2.698035e+09   1.000049
47  Redis02  redis-Redis_6379_Redis  (total_commands_proces... 2021-03-07 13:29:00+08:00 2021-03-07 13:30:00+08:00           2      3.743398e+09  3.743224e+09   1.000046
48  Redis02             redis-Redis_6379_Redis  (used_cpu_sys) 2021-03-07 13:28:00+08:00 2021-03-07 13:29:00+08:00           2      4.531299e+04  4.531100e+04   1.000044
49  Redis02            redis-Redis_6379_Redis  (used_cpu_user) 2021-03-07 13:28:00+08:00 2021-03-07 13:29:00+08:00           2      1.653288e+04  1.653236e+04   1.000032

[50 rows x 8 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: metric_df, metric_agg
# Ensure metric_df has timezone-aware timestamp column
if '_ts_tz' not in metric_df.columns:
    metric_df['_ts_tz'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    metric_df['_ts_tz'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure metric_agg has p95_value numeric and name column corresponds to kpi_name
metric_agg['p95_value'] = pd.to_numeric(metric_agg['p95_value'], errors='coerce')

# Candidate components
components = ["Mysql01","Mysql02","IG02","Redis02","MG01","Tomcat01","Tomcat02","Tomcat03","Tomcat04"]

# Build p95 lookup (global thresholds) - rule: compute thresholds before filtering window (we use metric_agg)
p95_lookup = metric_agg[['cmdb_id','name','p95_value']].rename(columns={'name':'kpi_name'}).copy()

# Merge global p95 into the full metric dataframe (inner join keeps only KPIs present in agg)
m_with_p95 = metric_df.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='inner')

# Incident window (inclusive) in UTC+8
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Filter to candidate components and incident window
m_win = m_with_p95[(m_with_p95['cmdb_id'].isin(components)) & (m_with_p95['_ts_tz'] >= window_start) & (m_with_p95['_ts_tz'] <= window_end)].copy()

# Mark anomalous points: value >= p95_value (require p95_value not null)
m_win['p95_value'] = pd.to_numeric(m_win['p95_value'], errors='coerce')
m_win['is_anom'] = m_win['p95_value'].notna() & (m_win['value'] >= m_win['p95_value'])

anom_points = m_win[m_win['is_anom']].copy()

# Compute per-point severity = value / p95_value if p95_value>0 else value
def point_sev(row):
    p = row['p95_value']
    v = row['value']
    try:
        if pd.isna(p) or p <= 0:
            return float(v)
        return float(v) / float(p)
    except:
        return float('nan')

if not anom_points.empty:
    anom_points['point_severity'] = anom_points.apply(point_sev, axis=1)
else:
    # create empty with expected columns
    anom_points = anom_points.copy()
    anom_points['point_severity'] = pd.Series(dtype=float)

# Group anomalous points by timestamp (minute) and cmdb_id
# Use the original integer 'timestamp' to ensure exact-minute grouping, but present tz-aware times for display
if not anom_points.empty:
    per_minute = anom_points.groupby(['timestamp','cmdb_id']).agg(
        anomaly_count_per_minute = ('kpi_name','nunique'),  # count distinct KPI groups anomalous that minute
        max_severity = ('point_severity','max')
    ).reset_index()
else:
    per_minute = pd.DataFrame(columns=['timestamp','cmdb_id','anomaly_count_per_minute','max_severity'])

# Convert timestamp to tz-aware datetime for display
if not per_minute.empty:
    per_minute['timestamp_tz'] = pd.to_datetime(per_minute['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Store full per-minute table in kernel variable
per_minute_anom_counts = per_minute.copy()

# Compute earliest timestamp when anomaly_count_per_minute >= 5 (if any)
if not per_minute_anom_counts.empty and (per_minute_anom_counts['anomaly_count_per_minute'] >= 5).any():
    earliest_ge5_row = per_minute_anom_counts[per_minute_anom_counts['anomaly_count_per_minute'] >= 5].sort_values(['timestamp']).iloc[0]
    earliest_ge5 = pd.to_datetime(earliest_ge5_row['timestamp'], unit='s', utc=True).tz_convert(tz)
else:
    earliest_ge5 = None

# Prepare compact output table: select columns and sort by timestamp asc then anomaly_count desc, limit top 50
if not per_minute_anom_counts.empty:
    out = per_minute_anom_counts.sort_values(['timestamp','anomaly_count_per_minute'], ascending=[True,False])
    out_display = out[['timestamp_tz','cmdb_id','anomaly_count_per_minute','max_severity']].rename(columns={'timestamp_tz':'timestamp'}).head(50)
else:
    out_display = per_minute_anom_counts.head(0)[['timestamp','cmdb_id','anomaly_count_per_minute','max_severity']]

# Final compact result variables to display
per_minute_anom_counts, out_display, earliest_ge5

```
Out[6]:
```
Summary of per-minute anomaly counts (window 2021-03-07 13:00–13:30 UTC+8):

- Earliest minute with >=5 distinct metric anomalies for a component: 2021-03-07 13:00:00+08:00.

- Peak minute highlights at 13:00+08 (highest anomaly counts / severities):
  - Mysql01: 114 anomalous KPI groups in that minute; max severity = 20.0 (many MySQL counters spiking >> P95).
  - Mysql02: 76 anomalous KPI groups in that minute; max severity = 1.00.
  - Redis02: 61 anomalous KPI groups in that minute; max severity ≈ 7.03.
  - MG01: 15 anomalous KPI groups in that minute; max severity ≈ 2.99.
  - IG02: 17 anomalous KPI groups in that minute; max severity ≈ 7.41.
  - Tomcat nodes and apache nodes also show dozens of per-minute anomalies at various minutes (see table).

- Full per-minute anomaly table (all minutes/components within the window) is stored in kernel variable: per_minute_anom_counts.

If you want, I can:
- show more rows (e.g., per_minute_anom_counts.head(100)), or
- plot a compact time series summary per component (text table) for the window.

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp   cmdb_id  anomaly_count_per_minute  max_severity              timestamp_tz
0    1615093200      IG02                        17      7.411438 2021-03-07 13:00:00+08:00
1    1615093200      MG01                        15      2.987111 2021-03-07 13:00:00+08:00
2    1615093200   Mysql01                       114     20.000000 2021-03-07 13:00:00+08:00
3    1615093200   Mysql02                        76      1.000000 2021-03-07 13:00:00+08:00
4    1615093200   Redis02                        61      7.028544 2021-03-07 13:00:00+08:00
..          ...       ...                       ...           ...                       ...
274  1615095000   Redis02                        39      1.000377 2021-03-07 13:30:00+08:00
275  1615095000  Tomcat01                        38      1.111111 2021-03-07 13:30:00+08:00
276  1615095000  Tomcat02                        23      1.000751 2021-03-07 13:30:00+08:00
277  1615095000  Tomcat03                        41      1.000752 2021-03-07 13:30:00+08:00
278  1615095000  Tomcat04                        34      1.002136 2021-03-07 13:30:00+08:00

[279 rows x 5 columns],                    timestamp   cmdb_id  anomaly_count_per_minute  max_severity
2  2021-03-07 13:00:00+08:00   Mysql01                       114     20.000000
3  2021-03-07 13:00:00+08:00   Mysql02                        76      1.000000
4  2021-03-07 13:00:00+08:00   Redis02                        61      7.028544
8  2021-03-07 13:00:00+08:00  Tomcat04                        42      1.043167
7  2021-03-07 13:00:00+08:00  Tomcat03                        24      1.004292
..                       ...       ...                       ...           ...
48 2021-03-07 13:05:00+08:00   Mysql02                       106      1.250000
47 2021-03-07 13:05:00+08:00   Mysql01                       101      1.000000
50 2021-03-07 13:05:00+08:00  Tomcat01                        33      1.000000
49 2021-03-07 13:05:00+08:00   Redis02                        31      1.000000
52 2021-03-07 13:05:00+08:00  Tomcat03                        25      1.000438

[50 rows x 4 columns], Timestamp('2021-03-07 13:00:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: trace_df, trace_agg
# Ensure trace_df has timezone-aware timestamp column
if '_ts_tz' not in trace_df.columns:
    trace_df['_ts_tz'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    trace_df['_ts_tz'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure trace_agg p95_value is numeric and its 'name' column corresponds to trace_name
trace_agg['p95_value'] = pd.to_numeric(trace_agg['p95_value'], errors='coerce')

# Allowed components
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Build p95 lookup (global thresholds computed from full series earlier)
p95_lookup = trace_agg[['cmdb_id','name','p95_value']].rename(columns={'name':'trace_name'}).copy()

# Merge global p95 into trace_df (inner join ensures only groups with global p95 are considered)
t = trace_df.merge(p95_lookup, on=['cmdb_id','trace_name'], how='inner')

# Incident window (inclusive) in UTC+8
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Filter to allowed components and window
t_win = t[(t['cmdb_id'].isin(components)) & (t['_ts_tz'] >= window_start) & (t['_ts_tz'] <= window_end)].copy()

# Mark anomalous points: value >= p95_value (require p95_value not null)
t_win['p95_value'] = pd.to_numeric(t_win['p95_value'], errors='coerce')
t_win['is_anom'] = t_win['p95_value'].notna() & (t_win['value'] >= t_win['p95_value'])

# Keep only anomalous points
t_anoms_points = t_win[t_win['is_anom']].copy()

# Aggregate per group (cmdb_id, trace_name)
if t_anoms_points.empty:
    trace_anoms_full = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity'])
else:
    agg = t_anoms_points.groupby(['cmdb_id','trace_name']).agg(
        earliest_anomaly_timestamp = ('_ts_tz', 'min'),
        anomaly_count_in_window    = ('value', 'count'),
        max_value_in_window       = ('value', 'max'),
        p95_value                 = ('p95_value', 'first')
    ).reset_index()
    # compute severity
    def compute_sev(row):
        p95 = row['p95_value']
        mx = row['max_value_in_window']
        try:
            if pd.isna(p95) or p95 == 0:
                return float(mx)
            return float(mx) / float(p95)
        except:
            return float('nan')
    agg['severity'] = agg.apply(compute_sev, axis=1)
    # Ensure timestamp tz-aware
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_timestamp']).dt.tz_convert(tz)
    trace_anoms_full = agg[['cmdb_id','trace_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity']].copy()

# Compute earliest overall trace anomaly timestamp among these groups (or None)
if not trace_anoms_full.empty:
    earliest_overall = trace_anoms_full['earliest_anomaly_timestamp'].min()
else:
    earliest_overall = None

# Prepare compact top-50 table sorted by earliest_anomaly_timestamp asc then severity desc
if not trace_anoms_full.empty:
    trace_anoms_top = trace_anoms_full.sort_values(['earliest_anomaly_timestamp','severity'], ascending=[True, False]).head(50)
else:
    trace_anoms_top = trace_anoms_full.head(0)

# Return compact results (store full per-group table in trace_anoms_full)
trace_anoms_top, earliest_overall

```
Out[7]:
```
Summary of trace anomaly detection (window 2021-03-07 13:00–13:30 UTC+8):

- Earliest trace anomaly among the inspected groups: 2021-03-07 13:00:00+08:00.

- Highest-severity trace anomalies (examples from the top results):
  - MG02 | trace.from_dockerB1.duration_mean — severity ≈ 1.91 (first at 13:00)
  - MG02 | trace.from_dockerA1.duration_mean — severity ≈ 1.15 (13:00)
  - MG02 | trace.to_dockerA2.row_count — severity ≈ 1.12 (13:00)
  - Tomcat01 | trace.self.row_count — severity ≈ 1.05 (13:00)
  - MG01 | trace.to_dockerA2.row_count — severity ≈ 1.04 (13:00)

- Pattern and interpretation:
  - Trace anomalies are concentrated at the start of the window (roughly 13:00–13:03 in the top rows).
  - Severity levels are modest (most in ~1.05–1.9 range); no large multi-fold (>2×) trace spikes among top groups.
  - Affected components include MG01/MG02 and Tomcat nodes primarily (also other allowed components in full results).

- Data availability:
  - Full per-group results are stored in kernel variable trace_anoms_full.
  - The earliest overall trace anomaly timestamp is 2021-03-07 13:00:00+08:00.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                         trace_name earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window    p95_value  severity
87       MG02  trace.from_dockerB1.duration_mean  2021-03-07 13:00:00+08:00                        2             0.193048     0.101265  1.906363
81       MG02  trace.from_dockerA1.duration_mean  2021-03-07 13:00:00+08:00                        2             0.134344     0.116585  1.152325
101      MG02        trace.to_dockerA2.row_count  2021-03-07 13:00:00+08:00                        2           157.000000   140.300000  1.119031
119  Tomcat01               trace.self.row_count  2021-03-07 13:00:00+08:00                        2          2642.000000  2517.750000  1.049350
62       MG01        trace.to_dockerA2.row_count  2021-03-07 13:00:00+08:00                        2           156.000000   150.050000  1.039653
..        ...                                ...                        ...                      ...                  ...          ...       ...
3        IG01    trace.to_Tomcat01.duration_mean  2021-03-07 13:03:00+08:00                        2             0.697512     0.603925  1.154965
108  Tomcat01      trace.from_IG01.duration_mean  2021-03-07 13:03:00+08:00                        2             0.697512     0.603925  1.154965
30       MG01  trace.from_Tomcat01.duration_mean  2021-03-07 13:03:00+08:00                        2             0.603349     0.523803  1.151863
120  Tomcat01        trace.to_MG01.duration_mean  2021-03-07 13:03:00+08:00                        2             0.603349     0.523803  1.151863
52       MG01   trace.from_dockerB2.duration_p95  2021-03-07 13:03:00+08:00                        2             1.012000     0.892637  1.133719

[50 rows x 7 columns], Timestamp('2021-03-07 13:00:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse variables from kernel: log_df, log_agg, error_logs_df
tz = pytz.timezone('Asia/Shanghai')

# Ensure log_df has timezone-aware timestamp column
if '_ts_tz' not in log_df.columns:
    log_df['_ts_tz'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    log_df['_ts_tz'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure global p95 in log_agg computed on full series (already in kernel). Normalize and numeric p95.
log_agg['p95_value'] = pd.to_numeric(log_agg['p95_value'], errors='coerce')

# Allowed components
components = ["Mysql01","Mysql02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Redis01","Redis02","apache01","apache02"]

# Build p95 lookup: name in log_agg corresponds to log_name
p95_lookup = log_agg[['cmdb_id','name','p95_value']].rename(columns={'name':'log_name'}).copy()

# Merge global p95 into log_df (inner join ensures we only consider groups with global p95 thresholds)
l = log_df.merge(p95_lookup, on=['cmdb_id','log_name'], how='inner')

# Incident window inclusive (UTC+8)
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Filter to allowed components and window
l_win = l[(l['cmdb_id'].isin(components)) & (l['_ts_tz'] >= window_start) & (l['_ts_tz'] <= window_end)].copy()

# Mark anomalous points: value >= p95_value (require p95_value not null)
l_win['p95_value'] = pd.to_numeric(l_win['p95_value'], errors='coerce')
l_win['is_anom'] = l_win['p95_value'].notna() & (l_win['value'] >= l_win['p95_value'])

# Keep only anomalous points
log_anom_points = l_win[l_win['is_anom']].copy()

# Aggregate per (cmdb_id, log_name)
if log_anom_points.empty:
    log_anoms_full = pd.DataFrame(columns=['cmdb_id','log_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity'])
else:
    agg = log_anom_points.groupby(['cmdb_id','log_name']).agg(
        earliest_anomaly_timestamp = ('_ts_tz', 'min'),
        anomaly_count_in_window    = ('value', 'count'),
        max_value_in_window       = ('value', 'max'),
        p95_value                 = ('p95_value', 'first')
    ).reset_index()
    # compute severity
    def compute_sev(row):
        p95 = row['p95_value']
        mx = row['max_value_in_window']
        try:
            if pd.isna(p95) or p95 == 0:
                return float(mx)
            return float(mx) / float(p95)
        except:
            return float('nan')
    agg['severity'] = agg.apply(compute_sev, axis=1)
    # ensure tz-aware timestamp
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_timestamp']).dt.tz_convert(tz)
    log_anoms_full = agg[['cmdb_id','log_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity']].copy()

# Compact top-20 table sorted by severity desc then anomaly_count desc
if not log_anoms_full.empty:
    log_anoms_top20 = log_anoms_full.sort_values(['severity','anomaly_count_in_window'], ascending=[False, False]).head(20)
else:
    log_anoms_top20 = log_anoms_full.head(0)

# Check error_logs.csv: count and earliest timestamp (if any)
# Ensure error_logs_df exists in kernel
if 'error_logs_df' in globals() and not error_logs_df.empty:
    # create tz-aware column
    if '_ts_tz' not in error_logs_df.columns:
        error_logs_df['_ts_tz'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    error_logs_count = int(error_logs_df.shape[0])
    error_logs_earliest = error_logs_df['_ts_tz'].min()
else:
    error_logs_count = 0
    error_logs_earliest = None

# Final compact outputs (stored full results in log_anoms_full)
log_anoms_top20, error_logs_count, error_logs_earliest

```
Out[8]:
```
Summary of log anomaly detection (window 2021-03-07 13:00–13:30 UTC+8):

- Highest-severity log anomaly:
  - Mysql01 | log.row_count — earliest 2021-03-07 13:00:00+08:00; 4 anomalous points; max=10.00 vs p95=5.00 → severity = 2.00.

- Other notable log anomalies (examples from top results):
  - Tomcat02 | log.gc_pause_max — earliest 13:06; 2 points; max=0.14 vs p95=0.1055 → severity ≈ 1.33.
  - Tomcat03 | log.row_count — earliest 13:01; 2 points; max=411 vs p95=378.5 → severity ≈ 1.09.
  - Tomcat04 | log.row_count — earliest 13:02; 3 points; max=389 vs p95=382 → severity ≈ 1.02.
  - apache02 / apache01 | log.row_count — small elevations (severity ≈ 1.04 and 1.00).

- Many "log.error_count" groups show high counts (e.g., 31 or 22 points) but their p95 is 0 in the dataset, producing severity = 0 by the implemented rule (p95==0). Those reflect nonzero occurrence counts but cannot be severity-scaled using p95 here.

- error_logs.csv:
  - Row count = 0; no raw error log rows found in the file for this window.

- Notes:
  - Full per-group log anomaly results are stored in kernel variable log_anoms_full.
  - The displayed top rows are the most relevant anomalies by severity; most log anomalies are modest (severity ~1–1.3) except the Mysql01 log.row_count spike (2.0).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  p95_value  severity
1    Mysql01     log.row_count  2021-03-07 13:00:00+08:00                        4                10.00     5.0000  2.000000
6   Tomcat02  log.gc_pause_max  2021-03-07 13:06:00+08:00                        2                 0.14     0.1055  1.327014
10  Tomcat03     log.row_count  2021-03-07 13:01:00+08:00                        2               411.00   378.5000  1.085865
9   Tomcat03  log.gc_pause_max  2021-03-07 13:18:00+08:00                        1                 0.11     0.1020  1.078431
7   Tomcat02     log.row_count  2021-03-07 13:13:00+08:00                        2               396.00   375.0000  1.056000
17  apache02     log.row_count  2021-03-07 13:15:00+08:00                        2               805.00   770.0000  1.045455
4   Tomcat01     log.row_count  2021-03-07 13:02:00+08:00                        2               390.00   381.0000  1.023622
13  Tomcat04     log.row_count  2021-03-07 13:02:00+08:00                        3               389.00   382.0000  1.018325
15  apache01     log.row_count  2021-03-07 13:01:00+08:00                        2               717.00   714.0000  1.004202
12  Tomcat04  log.gc_pause_max  2021-03-07 13:00:00+08:00                        5                 0.09     0.0900  1.000000
3   Tomcat01  log.gc_pause_max  2021-03-07 13:01:00+08:00                        2                 0.09     0.0900  1.000000
2   Tomcat01   log.error_count  2021-03-07 13:00:00+08:00                       31                 0.00     0.0000  0.000000
5   Tomcat02   log.error_count  2021-03-07 13:00:00+08:00                       31                 0.00     0.0000  0.000000
8   Tomcat03   log.error_count  2021-03-07 13:00:00+08:00                       31                 0.00     0.0000  0.000000
11  Tomcat04   log.error_count  2021-03-07 13:00:00+08:00                       31                 0.00     0.0000  0.000000
14  apache01   log.error_count  2021-03-07 13:00:00+08:00                       31                 0.00     0.0000  0.000000
16  apache02   log.error_count  2021-03-07 13:00:00+08:00                       31                 0.00     0.0000  0.000000
0    Mysql01   log.error_count  2021-03-07 13:00:00+08:00                       22                 0.00     0.0000  0.000000, 0, None)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cached variable if available; otherwise compute trace_anoms_full from trace_df & trace_agg
if 'trace_anoms_full' not in globals() or trace_anoms_full is None or trace_anoms_full.empty:
    # Fallback: compute using same logic as earlier (only if needed)
    # Ensure trace_df has tz-aware timestamp column
    if '_ts_tz' not in trace_df.columns:
        trace_df['_ts_tz'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        trace_df['_ts_tz'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    trace_agg['p95_value'] = pd.to_numeric(trace_agg['p95_value'], errors='coerce')
    p95_lookup = trace_agg[['cmdb_id','name','p95_value']].rename(columns={'name':'trace_name'}).copy()
    merged = trace_df.merge(p95_lookup, on=['cmdb_id','trace_name'], how='inner')
    window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
    window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)
    mwin = merged[(merged['_ts_tz'] >= window_start) & (merged['_ts_tz'] <= window_end)].copy()
    mwin['is_anom'] = mwin['p95_value'].notna() & (mwin['value'] >= mwin['p95_value'])
    t_points = mwin[mwin['is_anom']].copy()
    if t_points.empty:
        trace_anoms_full = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity'])
    else:
        agg = t_points.groupby(['cmdb_id','trace_name']).agg(
            earliest_anomaly_timestamp = ('_ts_tz', 'min'),
            anomaly_count_in_window    = ('value', 'count'),
            max_value_in_window       = ('value', 'max'),
            p95_value                 = ('p95_value', 'first')
        ).reset_index()
        def compute_sev(row):
            p95 = row['p95_value']; mx = row['max_value_in_window']
            if pd.isna(p95) or p95 == 0: return float(mx)
            return float(mx) / float(p95)
        agg['severity'] = agg.apply(compute_sev, axis=1)
        agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_timestamp']).dt.tz_convert(tz)
        trace_anoms_full = agg[['cmdb_id','trace_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_value','severity']].copy()

# Incident window (explicit)
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Build filter conditions:
# (a) cmdb_id in Mysql01 or Mysql02
# (b) trace_name contains substring 'Mysql' or 'to_Mysql' (case-insensitive)
mask_cmdb = trace_anoms_full['cmdb_id'].isin(['Mysql01','Mysql02'])
mask_name = trace_anoms_full['trace_name'].str.contains(r'mysql|to_mysql', case=False, na=False)
# Also ensure earliest_anomaly_timestamp within incident window (trace_anoms_full already computed for window but enforce)
mask_time = (trace_anoms_full['earliest_anomaly_timestamp'] >= window_start) & (trace_anoms_full['earliest_anomaly_timestamp'] <= window_end)

# Combined mask
mask = (mask_cmdb | mask_name) & mask_time

# Filtered table
trace_to_mysql_anoms = trace_anoms_full[mask].copy()

# Ensure columns/types
if not trace_to_mysql_anoms.empty:
    trace_to_mysql_anoms['earliest_anomaly_timestamp'] = pd.to_datetime(trace_to_mysql_anoms['earliest_anomaly_timestamp']).dt.tz_convert(tz)

# Compute count and earliest overall timestamp among matches
match_count = int(trace_to_mysql_anoms.shape[0])
earliest_overall = trace_to_mysql_anoms['earliest_anomaly_timestamp'].min() if match_count>0 else None

# Prepare compact top-50 sorted by earliest_anomaly_timestamp asc then severity desc
trace_to_mysql_anoms_top50 = trace_to_mysql_anoms.sort_values(['earliest_anomaly_timestamp','severity'], ascending=[True, False]).head(50)

# Display compact results: top50, count, earliest_overall
trace_to_mysql_anoms_top50, match_count, earliest_overall

```
Out[9]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use cached result if available
df = trace_to_mysql_anoms if 'trace_to_mysql_anoms' in globals() else pd.DataFrame()

match_count = int(df.shape[0])
earliest_overall = None
if match_count > 0:
    earliest_overall = pd.to_datetime(df['earliest_anomaly_timestamp']).min().tz_convert(tz)

summary_text = (
    "Trace→MySQL anomaly search summary (window 2021-03-07 13:00–13:30 UTC+8):\n\n"
    f"- Matching trace groups found: {match_count}\n"
    f"- Earliest anomaly among matches: {earliest_overall if earliest_overall is not None else None}\n\n"
    "Interpretation:\n"
    "- No trace anomaly groups involving MySQL (by component or by trace name) were detected in the incident window.\n"
    "- The filtered result table trace_to_mysql_anoms is empty."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, earliest_anomaly_timestamp, anomaly_count_in_window, max_value_in_window, p95_value, severity]
Index: [], 0, None)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: metric_agg, metric_anoms, metric_anom_points (if available)
# Ensure metric_agg exists
if 'metric_agg' not in globals():
    metric_agg = pd.DataFrame()  # fallback, but should exist from prior steps

# Normalize p95 numeric
if 'p95_value' in metric_agg.columns:
    metric_agg['p95_value'] = pd.to_numeric(metric_agg['p95_value'], errors='coerce')
else:
    metric_agg['p95_value'] = pd.Series(dtype=float)

# Candidate components
components = ["Tomcat01","Tomcat02","Tomcat03","Tomcat04","MG01","MG02","IG01","IG02",
              "apache01","apache02","Mysql01","Mysql02","Redis01","Redis02"]

# Substrings to match (case-insensitive)
substrs = ['request','requests','row_count','com','com_','queries','query','sessions',
           'connections','conn','total_commands','requests','request_count']

# Prepare lowercase name for matching
metric_agg['name_lc'] = metric_agg['name'].str.lower().fillna('')

# Filter by components and substring match
mask_comp = metric_agg['cmdb_id'].isin(components)
mask_sub = metric_agg['name_lc'].apply(lambda s: any(sub in s for sub in substrs))
matched = metric_agg[mask_comp & mask_sub].copy()

# Prepare output columns
matched = matched.rename(columns={'name':'kpi_name'})
matched = matched[['cmdb_id','kpi_name','count','p95_value']].copy()

# Determine if anomalous at target minute 2021-03-07 13:00:00+08:00
target_ts = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
target_epoch = int(pd.to_datetime(target_ts).tz_convert(pytz.utc).timestamp())  # epoch seconds UTC

# Build a lookup from metric_anoms if available (earliest_anomaly_timestamp)
anom_lookup = pd.DataFrame()
if 'metric_anoms' in globals() and isinstance(metric_anoms, pd.DataFrame) and not metric_anoms.empty:
    # ensure earliest_anomaly_timestamp tz-aware
    tmp = metric_anoms.copy()
    if tmp['earliest_anomaly_timestamp'].dtype == object:
        tmp['earliest_anomaly_timestamp'] = pd.to_datetime(tmp['earliest_anomaly_timestamp']).dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(tz) if tmp['earliest_anomaly_timestamp'].notna().any() else tmp['earliest_anomaly_timestamp']
    # normalize name column
    tmp = tmp[['cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window']].rename(columns={'name':'kpi_name'})
    anom_lookup = tmp

# Also check metric_anom_points if available for exact timestamp hits
points_lookup = pd.DataFrame()
if 'metric_anom_points' in globals() and isinstance(metric_anom_points, pd.DataFrame) and not metric_anom_points.empty:
    # metric_anom_points likely has integer 'timestamp' column in seconds
    points_lookup = metric_anom_points[['timestamp','cmdb_id','kpi_name']].copy()

# Function to check anomaly at target
def had_anomaly_at_target(row):
    cm = row['cmdb_id']; name = row['kpi_name']
    # Check anom_lookup exact earliest timestamp match (tz-aware)
    if not anom_lookup.empty:
        matches = anom_lookup[(anom_lookup['cmdb_id']==cm) & (anom_lookup['kpi_name']==name)]
        if not matches.empty:
            # compare minutes: convert to tz-aware and compare equality to target
            ets = pd.to_datetime(matches['earliest_anomaly_timestamp']).dt.tz_convert(tz)
            if (ets == target_ts).any():
                return True
            # also if anomaly_count_in_window>0 and earliest timestamp minute equals target
            if ((matches['anomaly_count_in_window']>0) & (ets == target_ts)).any():
                return True
    # Check point-level lookup for any anomalous point exactly at target epoch
    if not points_lookup.empty:
        pts = points_lookup[(points_lookup['timestamp']==target_epoch) & (points_lookup['cmdb_id']==cm) & (points_lookup['kpi_name']==name)]
        if not pts.empty:
            return True
    return False

# Apply check and add boolean column
if not matched.empty:
    matched['anomaly_at_2021-03-07_13:00'] = matched.apply(had_anomaly_at_target, axis=1)
else:
    matched['anomaly_at_2021-03-07_13:00'] = pd.Series(dtype=bool)

# Store in kernel variable
request_like_kpis = matched.copy()

# Sort and limit to up to 100 rows as requested
request_like_kpis_sorted = request_like_kpis.sort_values(['cmdb_id','p95_value'], ascending=[True, False]).head(100)

# Clean up helper column in metric_agg (avoid side effects)
metric_agg.drop(columns=['name_lc'], inplace=True, errors='ignore')

# Compact output: the dataframe and its shape
request_like_kpis_sorted, request_like_kpis_sorted.shape

```
Out[10]:
```
```python
# Compose a plain-English summary of the request/traffic-like KPI search results
import pandas as pd

# Use the cached filtered DataFrame
df = request_like_kpis_sorted if 'request_like_kpis_sorted' in globals() else (request_like_kpis if 'request_like_kpis' in globals() else pd.DataFrame())

total_groups = int(df.shape[0])
anoms_at_1300 = int(df['anomaly_at_2021-03-07_13:00'].sum()) if not df.empty else 0

# Sample examples (up to 6) for quick illustration
examples = []
if not df.empty:
    for _, r in df.head(6).iterrows():
        examples.append(f"{r['cmdb_id']} | {r['kpi_name']} | p95={r['p95_value']:.6g} | anomaly_at_13:00={bool(r['anomaly_at_2021-03-07_13:00'])}")

# Detect notable groups explicitly
mysql_maxconn_rows = df[(df['cmdb_id']=='Mysql01') & df['kpi_name'].str.contains('MaxConnections', case=False, na=False)]
mysql_maxconn_flag = bool(not mysql_maxconn_rows.empty and mysql_maxconn_rows.iloc[0]['anomaly_at_2021-03-07_13:00'])

# Count p95==0 groups among results (common for some session KPIs)
p95_zero_count = int((df['p95_value']==0).sum()) if not df.empty else 0

summary = (
    "Request/traffic-like KPI search summary (compact):\n\n"
    f"- Total matched KPI groups returned (preview up to 100): {total_groups}\n"
    f"- Groups showing an anomaly at 2021-03-07 13:00:00+08: {anoms_at_1300}\n\n"
    "Notable findings:\n"
    f"- Mysql01 contains a request-related KPI 'Mysql-MySQL_3306_MaxConnections' and it had an anomaly at 13:00:00+08: {mysql_maxconn_flag}.\n"
    f"- Several Tomcat 'Sessions' / session-related KPIs appear in the matches; many of these have p95==0 in the aggregated data (count with p95==0 in the preview: {p95_zero_count}), which limits severity normalization.\n"
    "- The matched KPIs span web/app (Tomcat, apache), gateway/ingress (IG/MG), DB (Mysql) and cache (Redis) components — indicating traffic/request-related signals are present across the stack.\n\n"
    "Examples (up to 6):\n"
    + ("\n".join(examples) if examples else "  (no examples available)") +
    "\n\nFull filtered list is cached in kernel variable: request_like_kpis (sorted preview in request_like_kpis_sorted)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count    p95_value  anomaly_at_2021-03-07_13:00
68        IG01    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum     21       1073.0                        False
156       IG02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum     20       1036.6                        False
248       MG01    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum     20       3430.0                        False
341       MG02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum      3       3390.8                        False
445    Mysql01                    Mysql-MySQL_3306_MaxConnections     20       2000.0                         True
...        ...                                                ...    ...          ...                          ...
1288  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31          0.0                         True
1289  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31          0.0                         True
1290  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31          0.0                         True
1291  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31          0.0                         True
1389  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  409074180.5                        False

[100 rows x 5 columns], (100, 5))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure request_like_kpis exists
if 'request_like_kpis' not in globals():
    request_like_kpis = pd.DataFrame(columns=['cmdb_id','kpi_name','p95_value'])

# Ensure metric_df has tz-aware timestamps
if '_ts_tz' not in metric_df.columns:
    metric_df['_ts_tz'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    metric_df['_ts_tz'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive)
window_start = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-07 13:30:00').tz_localize(tz)

# Target epoch for 2021-03-07 13:00:00+08 in UTC epoch seconds
target_ts = pd.to_datetime('2021-03-07 13:00:00').tz_localize(tz)
target_epoch = int(target_ts.tz_convert(pytz.UTC).timestamp())

# Prepare groups list with p95 from request_like_kpis
groups = request_like_kpis[['cmdb_id','kpi_name','p95_value']].drop_duplicates().copy()

# Merge groups into metric_df to extract timeseries for those groups
merged = metric_df.merge(groups, on=['cmdb_id','kpi_name'], how='inner')

# Filter to incident window
merged_win = merged[(merged['_ts_tz'] >= window_start) & (merged['_ts_tz'] <= window_end)].copy()

# Determine group-level anomaly at 13:00:00+08 by checking points at target_epoch
points_at_target = merged[(merged['timestamp'] == target_epoch)][['cmdb_id','kpi_name','value','p95_value']].copy()
if not points_at_target.empty:
    points_at_target['anomaly_flag'] = points_at_target['value'] >= points_at_target['p95_value']
    grp_flag = points_at_target.groupby(['cmdb_id','kpi_name'])['anomaly_flag'].any().reset_index().rename(columns={'anomaly_flag':'anomaly_at_2021_03_07_13_00'})
else:
    grp_flag = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_at_2021_03_07_13_00'])

# Merge the flag into per-minute timeseries
request_timeseries_full = merged_win[['timestamp','_ts_tz','cmdb_id','kpi_name','value']].merge(
    grp_flag, on=['cmdb_id','kpi_name'], how='left'
)

# Fill missing flags with False
request_timeseries_full['anomaly_at_2021_03_07_13_00'] = request_timeseries_full['anomaly_at_2021_03_07_13_00'].fillna(False)

# Sort by cmdb_id then timestamp
request_timeseries_full = request_timeseries_full.sort_values(['cmdb_id','timestamp']).reset_index(drop=True)

# Per-component summary: number_of_groups_in_request_like_list, groups_with_anomaly_at_13:00
comp_groups = groups.groupby('cmdb_id').agg(number_of_groups_in_request_like_list = ('kpi_name','nunique')).reset_index()
if not grp_flag.empty:
    comp_flags = grp_flag.groupby('cmdb_id').agg(groups_with_anomaly_at_13_00 = ('anomaly_at_2021_03_07_13_00','sum')).reset_index()
else:
    comp_flags = pd.DataFrame(columns=['cmdb_id','groups_with_anomaly_at_13_00'])
comp_summary = comp_groups.merge(comp_flags, on='cmdb_id', how='left')
comp_summary['groups_with_anomaly_at_13_00'] = comp_summary['groups_with_anomaly_at_13_00'].fillna(0).astype(int)
comp_summary = comp_summary.sort_values('cmdb_id').reset_index(drop=True)

# Store full extracted table in kernel variable
request_timeseries_full = request_timeseries_full

# Prepare compact preview (first 200 rows) and component summary for output
request_timeseries_preview = request_timeseries_full.head(200).rename(columns={'_ts_tz':'timestamp_tz'})
request_comp_summary = comp_summary

# Return compact outputs
request_timeseries_preview, request_comp_summary

```
Out[11]:
```
```python
# Compose a plain-English compact summary of the extracted request-like KPI time series
import pandas as pd

# Use cached variables if available
comp_summary = request_comp_summary if 'request_comp_summary' in globals() else pd.DataFrame()
ts_preview = request_timeseries_preview if 'request_timeseries_preview' in globals() else pd.DataFrame()
full_table_name = 'request_timeseries_full' if 'request_timeseries_full' in globals() else None

# Build numeric summary
if not comp_summary.empty:
    rows = []
    for _, r in comp_summary.iterrows():
        rows.append(f"{r['cmdb_id']}: {int(r['number_of_groups_in_request_like_list'])} groups, {int(r['groups_with_anomaly_at_13_00'])} groups anomalous at 13:00")
    comp_lines = "\n  - ".join(rows)
    total_groups = int(comp_summary['number_of_groups_in_request_like_list'].sum())
    total_with_anom = int(comp_summary['groups_with_anomaly_at_13_00'].sum())
else:
    comp_lines = "None"
    total_groups = 0
    total_with_anom = 0

summary_text = (
    "Request-like KPI time-series extraction (window 2021-03-07 13:00–13:30 UTC+8):\n\n"
    f"- Full per-minute time series for matched request/traffic KPIs has been extracted and saved in kernel variable: {full_table_name}.\n\n"
    f"- Component-level summary (groups / groups anomalous at 13:00):\n  - {comp_lines}\n\n"
    f"- Totals: {total_groups} matched KPI groups across components; {total_with_anom} of those groups had a value >= their global P95 at 13:00.\n\n"
    "Key observations:\n"
    "- MySQL: Mysql01 and Mysql02 each have ~26 request-like KPI groups; many of these were anomalous at 13:00 (Mysql01: 19/26, Mysql02: 15/26), indicating widespread DB-side spikes at the start of the window.\n"
    "- Tomcat: Each Tomcat node has ~14–17 request/session KPIs and most of those had anomalies at 13:00 (e.g., Tomcat01: 11/14, Tomcat02/03/04: 14/17), suggesting high request/session activity across app servers at the same minute.\n"
    "- Redis: Both Redis nodes show a smaller set of request-like KPIs (6 each) with a few anomalies at 13:00 (3 each).\n"
    "- IG/MG/apache: IG/MG and apache components have few matched groups and none flagged anomalous at 13:00 in the extracted set.\n\n"
    "Example preview (first rows of the extracted series are available in request_timeseries_preview):\n"
    f"{ts_preview.head(6).to_string(index=False) if not ts_preview.empty else '(no preview available)'}\n\n"
    "Next steps you might request:\n"
    "- Drill into specific high-count groups (e.g., Mysql01 groups) to see per-KPI timelines or correlate with trace/log anomalies.\n"
    "- Inspect the full request_timeseries_full table for additional minutes or export specific KPI series for further analysis.\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp              timestamp_tz  cmdb_id                                         kpi_name   value  anomaly_at_2021_03_07_13_00
0    1615093200 2021-03-07 13:00:00+08:00     IG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1005.0                        False
1    1615093320 2021-03-07 13:02:00+08:00     IG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1081.0                        False
2    1615093380 2021-03-07 13:03:00+08:00     IG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum   983.0                        False
3    1615093500 2021-03-07 13:05:00+08:00     IG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum   908.0                        False
4    1615093560 2021-03-07 13:06:00+08:00     IG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1002.0                        False
..          ...                       ...      ...                                              ...     ...                          ...
195  1615093680 2021-03-07 13:08:00+08:00  Mysql01                Mysql-MySQL_3306_Aborted Connects     0.0                         True
196  1615093680 2021-03-07 13:08:00+08:00  Mysql01                      Mysql-MySQL_3306_Com Delete     0.0                         True
197  1615093680 2021-03-07 13:08:00+08:00  Mysql01                Mysql-MySQL_3306_Com Delete Multi     0.0                         True
198  1615093680 2021-03-07 13:08:00+08:00  Mysql01                      Mysql-MySQL_3306_Com Insert     0.0                         True
199  1615093680 2021-03-07 13:08:00+08:00  Mysql01               Mysql-MySQL_3306_Com Insert Select     0.0                         True

[200 rows x 6 columns],      cmdb_id  number_of_groups_in_request_like_list  groups_with_anomaly_at_13_00
0       IG01                                      1                             0
1       IG02                                      1                             0
2       MG01                                      1                             0
3       MG02                                      1                             0
4    Mysql01                                     26                            19
5    Mysql02                                     26                            15
6    Redis01                                      6                             3
7    Redis02                                      6                             3
8   Tomcat01                                     14                            11
9   Tomcat02                                     17                            14
10  Tomcat03                                     17                            14
11  Tomcat04                                     17                            14
12  apache01                                      1                             0
13  apache02                                      1                             0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```